# BFCL-India v3 Training (Kaggle T4 x2)

- Base: Qwen2.5-3B-Instruct, fp16 (no quantization)
- LoRA r=16, alpha=16, all proj layers
- Compact tool schemas (0% truncation at 2048 tokens)
- eval_strategy=no (eval separately with eval.py after training)

In [ ]:
!pip install -q peft trl accelerate transformers datasets
!nvidia-smi

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.float16, device_map="auto", trust_remote_code=True
)
model.config.use_cache = False
print("Model loaded OK")

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0, bias="none", task_type=TaskType.CAUSAL_LM
))
model.print_trainable_parameters()

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={"train": "data/train.jsonl", "val": "data/val.jsonl"})
def fmt(ex): return {"text": [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in ex["messages"]]}
dataset = dataset.map(fmt, batched=True)
def tok(ex): return tokenizer(ex["text"], truncation=True, max_length=2048, padding=False)
train_ds = dataset["train"].map(tok, batched=True, remove_columns=dataset["train"].column_names)
val_ds = dataset["val"].map(tok, batched=True, remove_columns=dataset["val"].column_names)
print("Train:", len(train_ds), "Val:", len(val_ds))

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model, processing_class=tokenizer,
    train_dataset=train_ds, eval_dataset=val_ds,
    args=SFTConfig(
        output_dir="outputs",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=50,
        num_train_epochs=1,
        learning_rate=1e-4,
        fp16=True,
        logging_steps=25,
        eval_strategy="no",
        save_strategy="steps",
        save_steps=200,
        save_total_limit=2,
        optim="adamw_torch",
        seed=3407,
        report_to="none",
        gradient_checkpointing=True,
    ),
)
trainer.train()
print("Training complete!")

In [ ]:
model.save_pretrained("v2_lora_adapter")
tokenizer.save_pretrained("v2_lora_adapter")
print("LoRA adapter saved")